### testing mistral 7B model

In [1]:
!pip install -U transformers accelerate bitsandbytes peft datasets einops torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 23.6 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset, DatasetDict, Dataset, load_from_disk

In [ ]:
dataset = load_dataset("Hypersniper/riddles_v1", split="all")

README.md: 0.00B [00:00, ?B/s]

riddles.json: 0.00B [00:00, ?B/s]

riddles_2.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/469 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['answer', 'output', 'instruction'],
    num_rows: 469
})

In [ ]:
dataset[0]

{'answer': 'music',
 'output': "The question seems to be a riddle that is focusing on an entity that can evoke a range of emotional responses. The given clues are that it can make someone dream or stamp their feet, and it can also make someone laugh or weep. \n\nLet's break down these clues. Dreaming and stamping feet are both reactions that can be associated with feelings of joy, excitement, or anticipation. On the other hand, laughing and weeping are expressions of happiness and sadness respectively. So, we are looking for something that can induce these varied emotional reactions.\n\nConsidering these clues, one plausible answer could be 'music'. Here's why: \n\nMusic has a profound impact on our emotions. It has the power to uplift our spirits, soothe our nerves, make us feel happy, sad, excited, calm, and even lead us into a state of introspection or dreaming. \n\nWhen we listen to upbeat music, it often makes us want to move or dance. Hence, the reference to 'stamp their feet'. I

understanding BitsAndBytesConfig: https://medium.com/@anitha6g/bitsandbytesconfig-simplifying-quantization-for-efficient-large-language-models-3e482245fbd8

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype="float16", bnb_4bit_use_double_quant=True)

In [ ]:
import torch

In [ ]:
from google.colab import userdata

HF_token = userdata.get("HF_TOKEN")

In [ ]:
from huggingface_hub import login
login(token=HF_token)

In [ ]:
!pip install hf_transfer

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

few errors I faced, got resolved from here: https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/discussions/93

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("mistralai/Mistral-7B-Instruct-v0.2", resume_download=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

pytorch_model-00001-of-00003.bin:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

'/root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.2/snapshots/63a8b081895390a26e140280378bc85ec8bce07a'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    local_files_only=True
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
tokenizer_mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")

In [ ]:
tokenizer_mistral("Hello world, how are you?")

{'input_ids': [1, 22557, 1526, 28725, 910, 460, 368, 28804], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer_mistral.decode(tokenizer_mistral("Hello world, how are you?")["input_ids"])

'<s> Hello world, how are you?'

In [ ]:
prompt_template = """

Below are two riddles. Come up with 10 more. Output just the riddles, no numbering. Don't output anything else.

Riddles: \n\n {questions}

"""

In [ ]:
question_sample = dataset[0:2]["instruction"]

In [ ]:
question_sample

['At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep.',
 "Take off my skin, I won't cry, but you will."]

In [ ]:
prompt = prompt_template.format(questions="\n\n".join(question_sample))

In [ ]:
prompt

"\n\nBelow are two riddles. Come up with 10 more. Output just the riddles, no numbering. Don't output anything else.\n\nRiddles: \n\n At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep.\n\nTake off my skin, I won't cry, but you will.\n\n"

In [ ]:
print(prompt)



Below are two riddles. Come up with 10 more. Output just the riddles, no numbering. Don't output anything else.

Riddles: 

 At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep.

Take off my skin, I won't cry, but you will.




In [ ]:
messages = [
    {"role": "user", "content": prompt}
]

In [ ]:
input_tokens = tokenizer_mistral.apply_chat_template(messages, return_tensors="pt")

In [ ]:
input_tokens

tensor([[    1,   733, 16289, 28793, 28705,    13,    13, 20548,   336,   460,
           989,   408,  2558,   867, 28723, 12578,   582,   395, 28705, 28740,
         28734,   680, 28723, 15985,   776,   272,   408,  2558,   867, 28725,
           708,  1474,   288, 28723,  3189, 28742, 28707,  3825,  2424,  1112,
         28723,    13,    13, 28754,  2558,   867, 28747, 28705,    13,    13,
          1794,   272,  2622,   302,   528, 28725,   624,   993,  4999,   442,
         22665,   652,  4051, 28725,  1794,   272,  2622,   302,   528, 28725,
           624,   993,  5763,   442,  4662,   478,   615, 28723,    13,    13,
         19850,   805,   586,  4759, 28725,   315,  1747, 28742, 28707,  7843,
         28725,   562,   368,   622, 28723,    13,    13,   733, 28748, 16289,
         28793]])

In [ ]:
output_tokens = model.generate(input_tokens)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [ ]:
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model.device

device(type='cuda', index=0)

error fix ref.: https://stackoverflow.com/questions/70262173/error-expected-all-tensors-to-be-on-the-same-device-while-all-are-on-same-devi

In [ ]:
input_tokens = input_tokens.to(model.device)

output_tokens = model.generate(
        input_tokens,
        max_new_tokens = 500,
        do_sample = True,
        pad_token_id = tokenizer_mistral.eos_token_id)

In [ ]:
output_tokens

tensor([[    1,   733, 16289, 28793, 28705,    13,    13, 20548,   336,   460,
           989,   408,  2558,   867, 28723, 12578,   582,   395, 28705, 28740,
         28734,   680, 28723, 15985,   776,   272,   408,  2558,   867, 28725,
           708,  1474,   288, 28723,  3189, 28742, 28707,  3825,  2424,  1112,
         28723,    13,    13, 28754,  2558,   867, 28747, 28705,    13,    13,
          1794,   272,  2622,   302,   528, 28725,   624,   993,  4999,   442,
         22665,   652,  4051, 28725,  1794,   272,  2622,   302,   528, 28725,
           624,   993,  5763,   442,  4662,   478,   615, 28723,    13,    13,
         19850,   805,   586,  4759, 28725,   315,  1747, 28742, 28707,  7843,
         28725,   562,   368,   622, 28723,    13,    13,   733, 28748, 16289,
         28793, 28705, 28740, 28723,   315, 28742, 28719,  2608,  1419,   970,
           272,  4376,  2368, 28742, 28707, 27882, 28725,   560,   264,  3199,
          6581, 28725,   315, 28742, 28719,  7918,  

In [ ]:
tokenizer_mistral.decode(output_tokens[0])

'<s> [INST] \n\nBelow are two riddles. Come up with 10 more. Output just the riddles, no numbering. Don\'t output anything else.\n\nRiddles: \n\n At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep.\n\nTake off my skin, I won\'t cry, but you will.\n\n [/INST] 1. I\'m often found where the sun doesn\'t shine, In a dark corner, I\'m hidden from sight, A riddle wrapped in a mystery, inside an enigma, That is the answer to the question, "What am I?"\n\n2. I am taken from a mine, and shut up in a wooden case, From which I am never released, and yet I am used by almost every person.\n\n3. What goes up but never comes down, and what goes down but never goes up?\n\n4. The more you take, the more you leave behind. What am I?\n\n5. What has keys but can\'t open locks?\n\n6. What makes a graving tool, but can\'t write?\n\n7. I speak without a mouth and hear without ears. I have no body, but I come alive with wind. What am I?\n\n8. What is so 

In [ ]:
tokenizer_mistral.decode(output_tokens[0], skip_special_tokens=True)

'[INST] \n\nBelow are two riddles. Come up with 10 more. Output just the riddles, no numbering. Don\'t output anything else.\n\nRiddles: \n\n At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep.\n\nTake off my skin, I won\'t cry, but you will.\n\n [/INST] 1. I\'m often found where the sun doesn\'t shine, In a dark corner, I\'m hidden from sight, A riddle wrapped in a mystery, inside an enigma, That is the answer to the question, "What am I?"\n\n2. I am taken from a mine, and shut up in a wooden case, From which I am never released, and yet I am used by almost every person.\n\n3. What goes up but never comes down, and what goes down but never goes up?\n\n4. The more you take, the more you leave behind. What am I?\n\n5. What has keys but can\'t open locks?\n\n6. What makes a graving tool, but can\'t write?\n\n7. I speak without a mouth and hear without ears. I have no body, but I come alive with wind. What am I?\n\n8. What is so frag

In [ ]:
decoded_output = tokenizer_mistral.decode(output_tokens[0], skip_special_tokens=True)
print(decoded_output)

[INST] 

Below are two riddles. Come up with 10 more. Output just the riddles, no numbering. Don't output anything else.

Riddles: 

 At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep.

Take off my skin, I won't cry, but you will.

 [/INST] 1. I'm often found where the sun doesn't shine, In a dark corner, I'm hidden from sight, A riddle wrapped in a mystery, inside an enigma, That is the answer to the question, "What am I?"

2. I am taken from a mine, and shut up in a wooden case, From which I am never released, and yet I am used by almost every person.

3. What goes up but never comes down, and what goes down but never goes up?

4. The more you take, the more you leave behind. What am I?

5. What has keys but can't open locks?

6. What makes a graving tool, but can't write?

7. I speak without a mouth and hear without ears. I have no body, but I come alive with wind. What am I?

8. What is so fragile that saying its name breaks 

In [ ]:
print(decoded_output[len(prompt):])

will.

 [/INST] 1. I'm often found where the sun doesn't shine, In a dark corner, I'm hidden from sight, A riddle wrapped in a mystery, inside an enigma, That is the answer to the question, "What am I?"

2. I am taken from a mine, and shut up in a wooden case, From which I am never released, and yet I am used by almost every person.

3. What goes up but never comes down, and what goes down but never goes up?

4. The more you take, the more you leave behind. What am I?

5. What has keys but can't open locks?

6. What makes a graving tool, but can't write?

7. I speak without a mouth and hear without ears. I have no body, but I come alive with wind. What am I?

8. What is so fragile that saying its name breaks it?

9. What can be broken, but not held in your hand?

10. What has a heart that doesn't pump blood, and where is the heart situated?


### fine tuning phi-2

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("https://raw.githubusercontent.com/tharani001/Finetuning-Phi2-on-custom-dataset/refs/heads/main/Dataset/amazon_product_details.csv")

In [5]:
df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [6]:
df_revised = df[['product_name', 'category', 'discounted_price',
                 'actual_price', 'rating', 'rating_count', 'about_product']].copy()

df_revised = df_revised.dropna(subset=['about_product']).reset_index(drop=True)

df_revised.head()

,product_name,category,discounted_price,actual_price,rating,rating_count,about_product
0,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",4.2,"24,269",High Compatibility : Compatible With iPhone 12...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,4.0,"43,994","Compatible with all Type C enabled devices, be..."
2,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...


In [7]:
def format_prompt(row):
  return (f"Product Name: {row['product_name']}\n"
            f"Category: {row['category']}\n"
            f"Discounted Price: {row['discounted_price']}\n"
            f"Actual Price: {row['actual_price']}\n"
            f"Rating: {row['rating']} ({row['rating_count']} reviews)\n"
            f"Generate a compelling product description:")

In [8]:
df_revised['instruction'] = "Generate a product description based on the following details."
df_revised['input'] = df_revised.apply(format_prompt, axis=1)
df_revised['output'] = df_revised['about_product']

In [9]:
df_final = df_revised[['instruction', 'input', 'output']]

In [10]:
df_final.head()

,instruction,input,output
0,Generate a product description based on the fo...,Product Name: Wayona Nylon Braided USB to Ligh...,High Compatibility : Compatible With iPhone 12...
1,Generate a product description based on the fo...,Product Name: Ambrane Unbreakable 60W / 3A Fas...,"Compatible with all Type C enabled devices, be..."
2,Generate a product description based on the fo...,Product Name: Sounce Fast Phone Charging Cable...,【 Fast Charger& Data Sync】-With built-in safet...
3,Generate a product description based on the fo...,Product Name: boAt Deuce USB 300 2 in 1 Type-C...,The boAt Deuce USB 300 2 in 1 cable is compati...
4,Generate a product description based on the fo...,Product Name: Portronics Konnect L 1.2M Fast C...,[CHARGE & SYNC FUNCTION]- This cable comes wit...


In [11]:
df_final.to_json("phi2_product_descriptions.jsonl", orient='records', lines=True)

In [12]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="phi2_product_descriptions.jsonl")

Generating train split: 0 examples [00:00, ? examples/s]

In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1465
    })
})

In [14]:
dataset = dataset["train"].train_test_split(test_size=0.1)

In [15]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1318
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 147
    })
})

In [16]:
dataset["train"][0]

{'instruction': 'Generate a product description based on the following details.',
 'input': 'Product Name: Fire-Boltt Phoenix Smart Watch with Bluetooth Calling 1.3",120+ Sports Modes, 240*240 PX High Res with SpO2, Heart Rate Monitoring & IP67 Rating\nCategory: Electronics|WearableTechnology|SmartWatches\nDiscounted Price: ₹1,999\nActual Price: ₹9,999\nRating: 4.3 (27,704 reviews)\nGenerate a compelling product description:',
 'output': 'Fire-Boltt is India\' No 1 Wearable Watch Brand Q122 by IDC Worldwide quarterly wearable device tracker Q122.【Bluetooth Calling Watch】- Fire-Boltt Phoenix enables you to make and receive calls directly from your watch via the built-in speaker and microphone. This smartwatch features a dial pad, option to access recent calls & sync your phone’s contacts.;【High Resolution Display】- Comes with a 1.3" TFT Color Full Touch Screen and a 240*240 Pixel High Resolution this watch is covered to flaunt the sleek and stylish look always.|【120+ Sports Modes】- Trac

In [17]:
templates = {
    "user": "<|im_start|>user\n{msg}<|im_end|>",
    "assistant": "<|im_start|>assistant\n{msg}<|im_end|>"
}

In [18]:
def format_conversation(example):
  user_prompt = templates["user"].format(msg=example["instruction"] + "\n" + example["input"])
  assistant_prompt = templates["assistant"].format(msg=example["output"])
  return {"text": user_prompt + "\n" + assistant_prompt}

In [19]:
format_conversation(dataset["train"][0])

{'text': '<|im_start|>user\nGenerate a product description based on the following details.\nProduct Name: Fire-Boltt Phoenix Smart Watch with Bluetooth Calling 1.3",120+ Sports Modes, 240*240 PX High Res with SpO2, Heart Rate Monitoring & IP67 Rating\nCategory: Electronics|WearableTechnology|SmartWatches\nDiscounted Price: ₹1,999\nActual Price: ₹9,999\nRating: 4.3 (27,704 reviews)\nGenerate a compelling product description:<|im_end|>\n<|im_start|>assistant\nFire-Boltt is India\' No 1 Wearable Watch Brand Q122 by IDC Worldwide quarterly wearable device tracker Q122.【Bluetooth Calling Watch】- Fire-Boltt Phoenix enables you to make and receive calls directly from your watch via the built-in speaker and microphone. This smartwatch features a dial pad, option to access recent calls & sync your phone’s contacts.;【High Resolution Display】- Comes with a 1.3" TFT Color Full Touch Screen and a 240*240 Pixel High Resolution this watch is covered to flaunt the sleek and stylish look always.|【120+ 

In [20]:
print(format_conversation(dataset["train"][0]))

{'text': '<|im_start|>user\nGenerate a product description based on the following details.\nProduct Name: Fire-Boltt Phoenix Smart Watch with Bluetooth Calling 1.3",120+ Sports Modes, 240*240 PX High Res with SpO2, Heart Rate Monitoring & IP67 Rating\nCategory: Electronics|WearableTechnology|SmartWatches\nDiscounted Price: ₹1,999\nActual Price: ₹9,999\nRating: 4.3 (27,704 reviews)\nGenerate a compelling product description:<|im_end|>\n<|im_start|>assistant\nFire-Boltt is India\' No 1 Wearable Watch Brand Q122 by IDC Worldwide quarterly wearable device tracker Q122.【Bluetooth Calling Watch】- Fire-Boltt Phoenix enables you to make and receive calls directly from your watch via the built-in speaker and microphone. This smartwatch features a dial pad, option to access recent calls & sync your phone’s contacts.;【High Resolution Display】- Comes with a 1.3" TFT Color Full Touch Screen and a 240*240 Pixel High Resolution this watch is covered to flaunt the sleek and stylish look always.|【120+ 

In [21]:
dataset = dataset.map(format_conversation)

Map:   0%|          | 0/1318 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

In [23]:
model_name = "microsoft/phi-2"

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [25]:
tokenizer.pad_token

'<|endoftext|>'

In [26]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [27]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [28]:
model = prepare_model_for_kbit_training(model)

In [29]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "dense"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [30]:
model = get_peft_model(model, lora_config)

In [31]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PhiForCausalLM(
      (model): PhiModel(
        (embed_tokens): Embedding(51200, 2560)
        (layers): ModuleList(
          (0-31): 32 x PhiDecoderLayer(
            (self_attn): PhiAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=2560, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2560, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4bi

In [32]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [33]:
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/1318 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

In [34]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1318
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 147
    })
})

In [35]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./phi2_amazon_products_lora",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    optim="paged_adamw_8bit"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [36]:
data_collator

DataCollatorForLanguageModeling(tokenizer=CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=Fa

In [37]:
training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=500,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,

In [38]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

In [39]:
trainer

In [40]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: term101112 (term101112-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


TrainOutput(global_step=83, training_loss=2.353457347456231, metrics={'train_runtime': 1224.9827, 'train_samples_per_second': 1.076, 'train_steps_per_second': 0.068, 'total_flos': 1.089377689141248e+16, 'train_loss': 2.353457347456231, 'epoch': 1.0})

In [41]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [42]:
base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    torch_dtype='auto',
    device_map=None,
).to("cuda")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [43]:
lora_model = PeftModel.from_pretrained(base_model, "./phi2_amazon_products_lora/checkpoint-83")

In [44]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PhiForCausalLM(
      (model): PhiModel(
        (embed_tokens): Embedding(51200, 2560)
        (layers): ModuleList(
          (0-31): 32 x PhiDecoderLayer(
            (self_attn): PhiAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=2560, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2560, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
         

In [45]:
merged_model = lora_model.merge_and_unload()  # merges LoRA into base weights
merged_model.save_pretrained("./phi2_amazon_prods_merged_model")
tokenizer.save_pretrained("./phi2_amazon_prods_merged_model")

('./phi2_amazon_prods_merged_model/tokenizer_config.json',
 './phi2_amazon_prods_merged_model/special_tokens_map.json',
 './phi2_amazon_prods_merged_model/vocab.json',
 './phi2_amazon_prods_merged_model/merges.txt',
 './phi2_amazon_prods_merged_model/added_tokens.json',
 './phi2_amazon_prods_merged_model/tokenizer.json')

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./phi2_amazon_prods_merged_model", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("./phi2_amazon_prods_merged_model")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [47]:
tokenizer("Hello world, how are you?")

{'input_ids': [15496, 995, 11, 703, 389, 345, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [48]:
tokenizer.decode(tokenizer("Hello world, how are you?")["input_ids"])

'Hello world, how are you?'

In [49]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [50]:
prompt = "Write a short description for this Amazon product: Wireless Bluetooth headphones."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [51]:
model.eval()

with torch.no_grad():
  outputs = model.generate(
      **inputs,
      max_new_tokens=100,   # how many tokens to generate
      temperature=0.7,      # creativity
      top_p=0.9,            # nucleus sampling
      do_sample=True        # enables sampling instead of greedy
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Write a short description for this Amazon product: Wireless Bluetooth headphones.
Output: Enjoy wireless freedom and high-quality sound with these wireless Bluetooth headphones. They have a sleek design, a long battery life, and a built-in microphone for hands-free calls.



In [3]:
def generate_product_description(model, tokenizer,
                                 product_name, category, discounted_price,
                                 actual_price, rating, rating_count,
                                 max_new_tokens=150, temperature=0.7, top_p=0.9):
  """
  Generates a compelling product description based on your fine-tuned model.
  """
  prompt = f"""<|im_start|>user
      Generate a product description based on the following details.
      Product Name: {product_name}
      Category: {category}
      Discounted Price: {discounted_price}
      Actual Price: {actual_price}
      Rating: {rating} ({rating_count} reviews)
      Generate a compelling product description:
      <|im_end|>
      <|im_start|>assistant
    """

  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

  outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      temperature=temperature,
      top_p=top_p,
      do_sample=True,
      eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>")
  )

  generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return generated_text.strip()

In [17]:
products = [
  {
    "product_name": "The Midnight Library by Matt Haig",
    "category": "Books",
    "discounted_price": "$12",
    "actual_price": "$18",
    "rating": "4.7",
    "rating_count": "4300"
  },
  {
    "product_name": "Kindle Paperwhite",
    "category": "Electronics",
    "discounted_price": "$119",
    "actual_price": "$149",
    "rating": "4.8",
    "rating_count": "5600"
  }
]

In [19]:
import textwrap

def wrap_text(text, width=80):
  """
  Wraps a long string into multiple lines with a specified width.

  Parameters:
      text (str): The text to wrap.
      width (int): Maximum number of characters per line.

  Returns:
      str: Wrapped text.
  """
  return "\n".join(textwrap.wrap(text, width=width))

In [21]:
for p in products:
  desc = generate_product_description(
      model, tokenizer,
      p["product_name"], p["category"], p["discounted_price"],
      p["actual_price"], p["rating"], p["rating_count"]
  )
  wrapped_desc = wrap_text(desc, width=80)  # wrap at 80 characters
  print(f"\n--- {p['product_name']} ---\n{wrapped_desc}\n")
  print("\n\n\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- The Midnight Library by Matt Haig ---
<|im_start|>user       Generate a product description based on the following
details.       Product Name: The Midnight Library by Matt Haig       Category:
Books       Discounted Price: $12       Actual Price: $18       Rating: 4.7
(4300 reviews)       Generate a compelling product description:       <|im_end|>
<|im_start|>assistant     This product is a compelling read for anyone who is
looking to expand their horizons, learn new things, and change the way they
think about life.|>     <|im_end|> <|im_start|>assistant The Midnight Library by
Matt Haig is a beautiful and thought-provoking book. It is a must-read for
anyone who is looking to expand their horizons, learn new things, and change the
way they think about life.|> <|im_end|> <|im_start|>assistant Read this book to
discover what the real Midnight Library is and what it means to live your life
on purpose.|> <|im_end






--- Kindle Paperwhite ---
<|im_start|>user       Generate a produ

In [22]:
print(wrap_text(generate_product_description(model, tokenizer, "Yeti Rambler Tumbler 30oz", "Outdoors and Travel", "$34", "$39", "4.8", "1500"), 80))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|im_start|>user       Generate a product description based on the following
details.       Product Name: Yeti Rambler Tumbler 30oz       Category: Outdoors
and Travel       Discounted Price: $34       Actual Price: $39       Rating: 4.8
(1500 reviews)       Generate a compelling product description:       <|im_end|>
<|im_start|>assistant     This sleek and sturdy Yeti Rambler Tumbler 30oz is
designed to keep your beverages cold all day. It has a wide mouth opening for
easy filling and drinking, and it's BPA-free for safe use. The wide mouth
opening also makes it easy to fill and clean the tumbler. The tumbler is made
from durable and high-quality stainless steel for long-lasting use. Enjoy your
favorite beverages in style with this Yeti Rambler Tumbler 30oz.|<|im_end|>
<|im_start|>assistant     This yeti tumbler is made from premium stainless
steel, which is known for its durability and high-quality. The t


In [23]:
print(wrap_text(generate_product_description(model, tokenizer, "Organic Lavender Essential Oil", "Personal Care", "$12", "$18", "4.7", "850"), 80))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|im_start|>user       Generate a product description based on the following
details.       Product Name: Organic Lavender Essential Oil       Category:
Personal Care       Discounted Price: $12       Actual Price: $18       Rating:
4.7 (850 reviews)       Generate a compelling product description:
<|im_end|>       <|im_start|>assistant     Organic Lavender Essential Oil is a
pure and natural product. It contains pure Lavender essential oil which is made
from Lavender flower and is extracted using the steam distillation method. This
product is free from synthetic additives and has a light, floral fragrance. It
is recommended for people who prefer natural products.|<|im_end|>
<|im_start|>assistant     Lavender essential oil is known for its calming and
soothing effects. It helps to relieve stress and anxiety, promote relaxation,
and improve sleep quality. It also has antiseptic and antibacterial properties
which make it useful for treating skin irritations, wounds, and infections. This


In [24]:
print(wrap_text(generate_product_description(model, tokenizer, "LEGO Star Wars Millennium Falcon", "Toys & Games", "$159", "$179", "4.9", "2200"), 80))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|im_start|>user       Generate a product description based on the following
details.       Product Name: LEGO Star Wars Millennium Falcon       Category:
Toys & Games       Discounted Price: $159       Actual Price: $179       Rating:
4.9 (2200 reviews)       Generate a compelling product description:
<|im_end|>       <|im_start|>assistant     This LEGO Star Wars Millennium Falcon
set is a fun and engaging toy for kids aged 3 to 10 years. Made from ABS plastic
and with a realistic design, it includes 1,000 pieces including a Millennium
Falcon, Millennium Falcon cockpit, Millennium Falcon interior, Millennium Falcon
crew, Millennium Falcon engines, Millennium Falcon cockpit windows, Millennium
Falcon exterior, Millennium Falcon exterior panels, Millennium Falcon exterior
shell, Millennium Falcon interior door, Millennium Falcon interior panel,
Millennium Falcon interior windows, Millennium Falcon interior shell, Millennium
Falcon cockpit windows, Millennium Falcon cockpit panel, Millen

In [25]:
print(wrap_text(generate_product_description(model, tokenizer, "Adidas Predator Soccer Cleats", "Sports Footwear", "$89", "$120", "4.6", "1100"), 80))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|im_start|>user       Generate a product description based on the following
details.       Product Name: Adidas Predator Soccer Cleats       Category:
Sports Footwear       Discounted Price: $89       Actual Price: $120
Rating: 4.6 (1100 reviews)       Generate a compelling product description:
<|im_end|>       <|im_start|>assistant     These Adidas Predator Soccer Cleats
are a complete package for any soccer player. Featuring a durable synthetic
upper for long-lasting performance, these cleats provide a great grip on the
field, ensuring that you never miss a shot. The cleats come in a variety of
sizes to fit any foot and are suitable for both indoor and outdoor use. With a
lifetime warranty on the upper material, you can feel confident about the
quality of these cleats.|<|im_end|>       <|im_start|>assistant     These cleats
are available in different colors. Pick the color that matches your personality
and style. Choose the size that fits you the best.|<|im_end|>       <|


In [27]:
print(wrap_text(generate_product_description(model, tokenizer, "Bialetti Moka Express Coffee Maker", "Kitchen Appliances", "$39", "$49", "4.6", "1200"), 80))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|im_start|>user       Generate a product description based on the following
details.       Product Name: Bialetti Moka Express Coffee Maker       Category:
Kitchen Appliances       Discounted Price: $39       Actual Price: $49
Rating: 4.6 (1200 reviews)       Generate a compelling product description:
<|im_end|>       <|im_start|>assistant     This stylish and compact coffee maker
is perfect for small kitchens and offices. It brews the perfect cup of coffee in
minutes, with its 1.2 litre capacity and automatic brewing feature. The compact
design makes it ideal for those with limited counter space, while the sleek
chrome finish adds a touch of elegance to any kitchen.|<|im_end|>
<|im_start|>assistant     The Moka Express coffee maker is equipped with a
stainless steel filter basket and a glass carafe, which allows you to see the
coffee brewing process. It also has an automatic shut-off feature, which ensures
that your coffee maker stops brewing after the coffee is ready, saving you
ene

In [14]:
!zip -r /content/phi2_amazon_prods_merged_model.zip /content/phi2_amazon_prods_merged_model

  adding: content/phi2_amazon_prods_merged_model/ (stored 0%)
  adding: content/phi2_amazon_prods_merged_model/model-00002-of-00002.safetensors (deflated 8%)
  adding: content/phi2_amazon_prods_merged_model/tokenizer.json (deflated 82%)
  adding: content/phi2_amazon_prods_merged_model/vocab.json (deflated 59%)
  adding: content/phi2_amazon_prods_merged_model/special_tokens_map.json (deflated 75%)
  adding: content/phi2_amazon_prods_merged_model/merges.txt (deflated 53%)
  adding: content/phi2_amazon_prods_merged_model/generation_config.json (deflated 24%)
  adding: content/phi2_amazon_prods_merged_model/added_tokens.json (deflated 84%)
  adding: content/phi2_amazon_prods_merged_model/model.safetensors.index.json (deflated 96%)
  adding: content/phi2_amazon_prods_merged_model/config.json (deflated 48%)
  adding: content/phi2_amazon_prods_merged_model/tokenizer_config.json (deflated 94%)
  adding: content/phi2_amazon_prods_merged_model/model-00001-of-00002.safetensors (deflated 8%)


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [4]:
base_model_name = "microsoft/phi-2"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
base_model.device

device(type='cuda', index=0)

In [5]:
if base_tokenizer.pad_token is None:
  base_tokenizer.pad_token = base_tokenizer.eos_token

In [8]:
product_name = "Bialetti Moka Express Coffee Maker"
category = "Kitchen Appliances"
discounted_price = "$39"
actual_price = "$49"
rating = "4.6"
rating_count = "1200"

prompt = (
    f"Generate a product description based on the following details.\n"
    f"Product Name: {product_name}\n"
    f"Category: {category}\n"
    f"Discounted Price: {discounted_price}\n"
    f"Actual Price: {actual_price}\n"
    f"Rating: {rating} ({rating_count} reviews)\n"
    f"Generate a compelling product description:"
)

In [9]:
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
base_model.eval()
with torch.no_grad():
  outputs = base_model.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=True,
      temperature=0.7,
      top_p=0.9,
      pad_token_id=base_tokenizer.eos_token_id
  )

In [13]:
base_tokenizer.decode(outputs[0])

"Generate a product description based on the following details.\nProduct Name: Bialetti Moka Express Coffee Maker\nCategory: Kitchen Appliances\nDiscounted Price: $39\nActual Price: $49\nRating: 4.6 (1200 reviews)\nGenerate a compelling product description: \nAnswer: The Bialetti Moka Express Coffee Maker is the perfect addition to any coffee lover's kitchen. With its sleek design and easy-to-use features, this coffee maker is a must-have for anyone who wants to enjoy a freshly brewed cup of coffee at home. \n\nThe Bialetti Moka Express Coffee Maker is a classic coffee maker that has been around for decades, and for good reason. With its unique brewing process, this coffee maker produces a rich and flavorful cup of coffee that is sure to impress. \n\nThe Bialetti Moka Express Coffee Maker is also incredibly easy to use. Simply fill the reservoir with water, add your coffee grounds, and press the button to start brewing."

In [15]:
import textwrap

In [16]:
text = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
wrapped_text = "\n".join(textwrap.wrap(text, width=80))

print(wrapped_text)

Generate a product description based on the following details. Product Name:
Bialetti Moka Express Coffee Maker Category: Kitchen Appliances Discounted
Price: $39 Actual Price: $49 Rating: 4.6 (1200 reviews) Generate a compelling
product description:  Answer: The Bialetti Moka Express Coffee Maker is the
perfect addition to any coffee lover's kitchen. With its sleek design and easy-
to-use features, this coffee maker is a must-have for anyone who wants to enjoy
a freshly brewed cup of coffee at home.   The Bialetti Moka Express Coffee Maker
is a classic coffee maker that has been around for decades, and for good reason.
With its unique brewing process, this coffee maker produces a rich and flavorful
cup of coffee that is sure to impress.   The Bialetti Moka Express Coffee Maker
is also incredibly easy to use. Simply fill the reservoir with water, add your
coffee grounds, and press the button to start brewing.


In [17]:
def test_base_model(product_name, category, discounted_price, actual_price, rating, rating_count, max_new_tokens=150, temperature=0.7, top_p=0.9, wrap_width=80):
  """
  Generate a product description using the base Phi-2 model.

  Parameters:
      product_name, category, discounted_price, actual_price, rating, rating_count: product details
      max_new_tokens: max tokens to generate
      temperature: sampling temperature
      top_p: nucleus sampling probability
      wrap_width: number of characters per line for wrapped output

  Returns:
      Wrapped string of the generated product description
  """
  prompt = (
      f"Generate a product description based on the following details.\n"
      f"Product Name: {product_name}\n"
      f"Category: {category}\n"
      f"Discounted Price: {discounted_price}\n"
      f"Actual Price: {actual_price}\n"
      f"Rating: {rating} ({rating_count} reviews)\n"
      f"Generate a compelling product description:"
  )

  inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
  base_model.eval()
  with torch.no_grad():
      outputs = base_model.generate(
          **inputs,
          max_new_tokens=max_new_tokens,
          do_sample=True,
          temperature=temperature,
          top_p=top_p,
          pad_token_id=base_tokenizer.eos_token_id
      )

  text = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
  return "\n".join(textwrap.wrap(text, width=wrap_width))

In [18]:
def wrap_text(text, width=80):
  return "\n".join(textwrap.wrap(text, width=width))

products = [
    ("Yeti Rambler Tumbler 30oz", "Outdoors and Travel", "$34", "$39", "4.8", "1500"),
    ("Organic Lavender Essential Oil", "Personal Care", "$12", "$18", "4.7", "850"),
    ("LEGO Star Wars Millennium Falcon", "Toys & Games", "$159", "$179", "4.9", "2200"),
    ("Adidas Predator Soccer Cleats", "Sports Footwear", "$89", "$120", "4.6", "1100"),
    ("Bialetti Moka Express Coffee Maker", "Kitchen Appliances", "$39", "$49", "4.6", "1200")
]

for p in products:
  desc = test_base_model(*p)
  print(f"\n--- {p[0]} ---\n{wrap_text(desc, 80)}\n\n")


--- Yeti Rambler Tumbler 30oz ---
Generate a product description based on the following details. Product Name:
Yeti Rambler Tumbler 30oz Category: Outdoors and Travel Discounted Price: $34
Actual Price: $39 Rating: 4.8 (1500 reviews) Generate a compelling product
description: Experience the perfect blend of style and functionality with the
Yeti Rambler Tumbler 30oz. This insulated tumbler is designed to keep your
beverages at the perfect temperature, whether you're on the go or enjoying a
leisurely picnic. With its durable construction and leak-proof lid, you can
trust that your drink will stay safe and secure. The Yeti Rambler Tumbler 30oz
is not only a practical choice, but it's also a stylish accessory that will
elevate your outdoor adventures.



--- Organic Lavender Essential Oil ---
Generate a product description based on the following details. Product Name:
Organic Lavender Essential Oil Category: Personal Care Discounted Price: $12
Actual Price: $18 Rating: 4.7 (850 reviews) G

fine tuned model: <br>
1. This stylish and compact coffee maker
is perfect for small kitchens and offices. It brews the perfect cup of coffee in
minutes, with its 1.2 litre capacity and automatic brewing feature. The compact
design makes it ideal for those with limited counter space, while the sleek
chrome finish adds a touch of elegance to any kitchen.|<|im_end|>
<|im_start|>assistant     The Moka Express coffee maker is equipped with a
stainless steel filter basket and a glass carafe, which allows you to see the
coffee brewing process. It also has an automatic shut-off feature, which ensures
that your coffee maker stops brewing after the coffee is ready, saving you
energy and preventing any potential accidents.

2. These Adidas Predator Soccer Cleats
are a complete package for any soccer player. Featuring a durable synthetic
upper for long-lasting performance, these cleats provide a great grip on the
field, ensuring that you never miss a shot. The cleats come in a variety of
sizes to fit any foot and are suitable for both indoor and outdoor use. With a
lifetime warranty on the upper material, you can feel confident about the
quality of these cleats.|<|im_end|>       <|im_start|>assistant     These cleats
are available in different colors. Pick the color that matches your personality
and style. Choose the size that fits you the best.


3. Organic Lavender Essential Oil is a
pure and natural product. It contains pure Lavender essential oil which is made
from Lavender flower and is extracted using the steam distillation method. This
product is free from synthetic additives and has a light, floral fragrance. It
is recommended for people who prefer natural products.|<|im_end|>
<|im_start|>assistant     Lavender essential oil is known for its calming and
soothing effects. It helps to relieve stress and anxiety, promote relaxation,
and improve sleep quality. It also has antiseptic and antibacterial properties
which make it useful for treating skin irritations, wounds, and infections. This
product is a perfect addition to your skincare routine

<br>

base model: <br>
1. The Bialetti Moka Express Coffee Maker is the
perfect addition to any coffee lover's kitchen. With its sleek design and easy-
to-use features, this coffee maker is a must-have for anyone who wants to enjoy
a freshly brewed cup of coffee at home.   The Bialetti Moka Express Coffee Maker
is a classic coffee maker that has been around for decades, and for good reason.
With its unique brewing process, this coffee maker produces a rich and flavorful
cup of coffee that is sure to impress.   The Bialetti Moka Express Coffee Maker
is also incredibly easy to use. Simply fill the reservoir with water, add your
coffee grounds, and press the button to start brewing.

2. Experience the perfect blend of style and performance with the
Adidas Predator Soccer Cleats. Crafted with cutting-edge technology, these
cleats are designed to enhance your on-field performance, providing excellent
traction and stability. With their sleek design and breathable upper, they are
not only functional but also fashionable. Get a 4.6-star rating and enjoy the
best of both worlds with the Adidas Predator Soccer Cleats.

3. Discover the soothing power of Organic Lavender Essential Oil with
this discounted price! Perfect for calming the mind and body, this essential oil
is a natural remedy for stress and relaxation. With its calming properties, it
can help promote better sleep and reduce anxiety. Try it out and experience the
benefits for yourself! #EssentialOil #Lavender #SoothingPower #Relaxation #CalmMind #NaturalRemedy #StressRelief #BetterSleep #AnxietyReduction #Discount #ScentedEssentialOil #ScentedProducts #EssentialOils #EssentialOilsForWellness #EssentialOilsForRelaxation #EssentialOilsForSleep #EssentialOilsForSt